# Solar Wind vs Geomagnetic Response

*Reproducible research notebook from [Flarient](https://flarient.com) — the space weather intelligence platform.*

**About this notebook:** This notebook is part of the [Flarient Research Notebooks](https://github.com/flarientglobal/flarient-notebooks) collection. It uses public data from NOAA SWPC, NASA, and the Flarient API.


## 1. Introduction

The relationship between solar wind parameters and geomagnetic activity is fundamental to space weather forecasting. In this notebook, we'll explore how solar wind speed and Bz affect the Kp index.


In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (12, 6)


## 2. Fetching Data

We'll fetch both Kp and solar wind data from NOAA SWPC.


In [ ]:
# Fetch Kp
kp_resp = requests.get("https://services.swpc.noaa.gov/json/planetary_k_index_1m.json", timeout=30)
kp_df = pd.DataFrame(kp_resp.json())
kp_df['time_tag'] = pd.to_datetime(kp_df['time_tag'])
kp_df['kp'] = kp_df['kp'].astype(float)
kp_df = kp_df.set_index('time_tag')

# Fetch solar wind
sw_resp = requests.get("https://services.swpc.noaa.gov/products/ace/ace_swepam_1m.json", timeout=30)
sw_data = sw_resp.json()
sw_df = pd.DataFrame(sw_data[1:], columns=sw_data[0])
sw_df['time_tag'] = pd.to_datetime(sw_df['time_tag'])
sw_df['speed'] = sw_df['speed'].astype(float)
sw_df['bz'] = sw_df['bz_gsm'].astype(float)
sw_df['density'] = sw_df['density'].astype(float)
sw_df = sw_df.set_index('time_tag')

# Merge
merged = kp_df.join(sw_df, how='inner')
print(f"Merged dataset: {len(merged)} readings")


## 3. Correlation Analysis

Let's examine the correlation between solar wind parameters and Kp.


In [ ]:
# Calculate correlations
correlations = merged[['kp', 'speed', 'bz', 'density']].corr()
print("Correlation Matrix:")
print(correlations.round(3))

# Key insight: Bz has a negative correlation with Kp (southward Bz → higher Kp)
print(f"\nCorrelation Kp vs Speed: {correlations.loc['kp', 'speed']:.3f}")
print(f"Correlation Kp vs Bz: {correlations.loc['kp', 'bz']:.3f}")
print(f"Correlation Kp vs Density: {correlations.loc['kp', 'density']:.3f}")


## 4. Visualising the Relationship

Let's create a scatter plot of Bz vs Kp, colored by solar wind speed.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(merged['bz'], merged['kp'], c=merged['speed'], 
                     cmap='viridis', alpha=0.3, s=10)
plt.colorbar(scatter, label='Solar Wind Speed (km/s)')
ax.set_xlabel('Bz (nT)')
ax.set_ylabel('Kp Index')
ax.set_title('Solar Wind Bz vs Kp Index — Source: NOAA SWPC')
ax.axvline(x=0, color='white', alpha=0.3)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig('bz_vs_kp.png', dpi=150, bbox_inches='tight')
plt.show()


## 5. Time Lag Analysis

Geomagnetic response to solar wind changes is not instantaneous. Let's check for time lags.


In [ ]:
# Cross-correlation between Bz and Kp
from scipy.signal import correlate

# Normalize and compute cross-correlation
bz_norm = (merged['bz'] - merged['bz'].mean()) / merged['bz'].std()
kp_norm = (merged['kp'] - merged['kp'].mean()) / merged['kp'].std()

# Limit to last 1000 points for speed
bz_arr = bz_norm.values[-1000:]
kp_arr = kp_norm.values[-1000:]

corr = correlate(kp_arr, bz_arr, mode='full')
lags = np.arange(-len(bz_arr) + 1, len(kp_arr))
max_lag = lags[np.argmax(np.abs(corr))]

print(f"Maximum correlation at lag: {max_lag} readings")
print(f"  (~{abs(max_lag) * 1} minutes lag)")

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(lags, corr, color='#22d3ee')
ax.set_xlabel('Lag (readings)')
ax.set_ylabel('Cross-correlation')
ax.set_title('Cross-correlation: Bz → Kp Response Time')
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()


## 6. Conclusion

This notebook demonstrated:
1. The strong relationship between southward Bz and geomagnetic activity
2. Solar wind speed as a secondary driver
3. The time lag between solar wind changes and Kp response

For real-time monitoring of these relationships, visit [flarient.com](https://flarient.com).


---

## About Flarient

[Flarient](https://flarient.com) is a space weather intelligence platform providing real-time data, forecasts, and community-driven observations. Visit [flarient.com](https://flarient.com) for live space weather conditions, aurora forecasts, and more.

## License

MIT — This notebook is open source. [View on GitHub](https://github.com/flarientglobal/flarient-notebooks).
